# Comparison of all setups in one convenient table (throttled)

In [1]:
import pandas as pd;
import numpy as np;
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
import scipy.stats as stats

plt.rcParams.update({'font.size': 14})
pd.set_option('display.float_format', lambda x: '%.2f' % x)
path = '../../../../playwright/results/core-web-vitals/testrun-9/'


In [2]:
features = ['navTime', 'totalTime', 'lcp', 'fcp', 'ttfb', 'tbt', 'tti', 'longestTask', 'longTasks', 'nf:init', 'nf:config','nf:loaded']

dirty_dfs =	{
  "Monolith": pd.read_csv(f'{path}results-monolith-throttled.csv', sep=',').iloc[5:],
  "CSR": pd.read_csv(f'{path}results-csr-throttled.csv', sep=',').iloc[5:],
  "CSR sd": pd.read_csv(f'{path}results-csr-throttled.csv', sep=',').iloc[5:],
  "SSRH": pd.read_csv(f'{path}results-ssrh-throttled.csv', sep=',').iloc[5:],
  "SSRH sd": pd.read_csv(f'{path}results-ssrh-sd-throttled.csv', sep=',').iloc[5:],
  "SSRV": pd.read_csv(f'{path}results-ssrv-sd.csv', sep=',').iloc[5:],
}

In [3]:
def detect_outliers(_df, _features, contamination=0.1):
    clf = IsolationForest(contamination=contamination, random_state=42)
    outliers = clf.fit_predict(_df[_features])
    return outliers == 1

masks = {}
dfs = {}
target_features = ['navTime', 'totalTime', 'lcp', 'fcp', 'ttfb']

for name, _df in dirty_dfs.items():
    mask = detect_outliers(_df, target_features)
    masks[name] = mask
    dfs[name] = _df[mask].copy()

In [4]:
columns = [ 'ttfb','fcp','nf:init','lcp','tti','nf:loaded','tbt','longestTask']
rows = []

for name, df in dfs.items():
    mean_row = df[columns].mean()
    rows.append((f"{name} (mean)", mean_row))

for name, df in dfs.items():
    percentile_row = df[columns].quantile(0.75)
    rows.append((f"{name} (75th)", percentile_row))

result_df = pd.DataFrame([row[1] for row in rows], index=[row[0] for row in rows])
result_df = result_df.mask(result_df < 0, '-')
result_df

,ttfb,fcp,nf:init,lcp,tti,nf:loaded,tbt,longestTask
Monolith (mean),189.99,979.35,-,979.35,979.35,-,0.00,-
CSR (mean),175.77,965.46,1176.26,4399.65,3619.52,3665.96,37.24,87.24
CSR sd (mean),175.77,965.46,1176.26,4399.65,3619.52,3665.96,37.24,87.24
SSRH (mean),205.06,1025.96,1317.97,1025.96,3475.10,3616.95,18.28,68.28
SSRH sd (mean),205.46,1027.83,1321.01,1042.66,5129.18,5139.74,36.10,86.10
SSRV (mean),28.51,82.27,-,91.85,82.27,-,0.00,-
Monolith (75th),190.30,984.40,-,984.40,984.40,-,0.00,-
CSR (75th),176.20,970.20,1181.30,4412.67,3628.97,3674.37,38.00,88.00
CSR sd (75th),176.20,970.20,1181.30,4412.67,3628.97,3674.37,38.00,88.00
SSRH (75th),205.80,1029.40,1323.50,1029.40,3482.05,3623.30,19.00,69.00
